In [1]:
import json
from pathlib import Path

import pandas as pd


# Locate the project directories.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "explore" else Path.cwd()
DETAILS_DIR = PROJECT_ROOT / "data" / "cache" / "details"

# Load all successfully cached movie details responses.
movie_records = []

for file_path in sorted(DETAILS_DIR.glob("*.json")):
    with file_path.open("r", encoding="utf-8") as file:
        response = json.load(file)

    # Extract the actual TMDb movie data from the cache wrapper.
    movie_records.append(response["data"])

# Convert the records into a pandas DataFrame.
films_df = pd.DataFrame(movie_records)

# Display the number of records and columns.
print(f"Total movie records: {len(films_df)}")
print(f"Total columns: {len(films_df.columns)}")


Total movie records: 112
Total columns: 27


In [2]:
# Display column names, data types and non-null counts.
films_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 112 entries, 0 to 111
Data columns (total 27 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  112 non-null    bool   
 1   backdrop_path          112 non-null    str    
 2   belongs_to_collection  96 non-null     object 
 3   budget                 112 non-null    int64  
 4   genres                 112 non-null    object 
 5   homepage               112 non-null    str    
 6   id                     112 non-null    int64  
 7   imdb_id                112 non-null    str    
 8   origin_country         112 non-null    object 
 9   original_language      112 non-null    str    
 10  original_title         112 non-null    str    
 11  overview               112 non-null    str    
 12  popularity             112 non-null    float64
 13  poster_path            112 non-null    str    
 14  production_companies   112 non-null    object 
 15  production_countr

In [3]:
# Select film-level fields relevant to licensing analysis.
film_columns = [
    "id",
    "title",
    "release_date",
    "belongs_to_collection",
    "budget",
    "revenue",
    "popularity",
    "vote_average",
    "vote_count",
]

# Create a focused analytical view without modifying the raw DataFrame.
films_analysis = films_df[film_columns].copy()

# Inspect the selected columns.
display(films_analysis.head())

,id,title,release_date,belongs_to_collection,budget,revenue,popularity,vote_average,vote_count
0,10138,Iron Man 2,2010-04-28,"{'id': 131292, 'name': 'Iron Man Collection', ...",200000000,623933331,21.1095,6.856,23254
1,10192,Shrek Forever After,2010-05-16,"{'id': 2150, 'name': 'Shrek Collection', 'post...",165000000,752600867,12.7559,6.400,8241
2,10193,Toy Story 3,2010-06-16,"{'id': 10194, 'name': 'Toy Story Collection', ...",200000000,1067316101,14.2103,7.818,16462
3,105,Back to the Future,1985-07-03,"{'id': 264, 'name': 'Back to the Future Collec...",19000000,391546852,25.1565,8.332,22791
4,10681,WALL·E,2008-06-26,None,180000000,521311860,19.0410,8.129,21213


In [4]:
# Extract collection identifiers from nested collection objects.
films_analysis["collection_id"] = films_analysis[
    "belongs_to_collection"
].apply(
    lambda collection: collection.get("id")
    if isinstance(collection, dict) else None
)

# Extract collection names.
films_analysis["collection_name"] = films_analysis[
    "belongs_to_collection"
].apply(
    lambda collection: collection.get("name")
    if isinstance(collection, dict) else None
)

# Count films with and without TMDb collection information.
with_collection = films_analysis["collection_id"].notna().sum()
without_collection = films_analysis["collection_id"].isna().sum()

print(f"Films with a collection: {with_collection}")
print(f"Films without a collection: {without_collection}")

# Display the distinct collections represented in the dataset.
collections = (
    films_analysis.loc[
        films_analysis["collection_id"].notna(),
        ["collection_id", "collection_name"]
    ]
    .drop_duplicates()
    .sort_values("collection_name")
)

display(collections)

Films with a collection: 96
Films without a collection: 16


,collection_id,collection_name
60,8091.0,Alien Collection
3,264.0,Back to the Future Collection
93,422837.0,Blade Runner Collection
18,1570.0,Die Hard Collection
79,2980.0,Ghostbusters Collection
12,1241.0,Harry Potter Collection
64,8354.0,Ice Age Collection
35,84.0,Indiana Jones Collection
0,131292.0,Iron Man Collection
73,2366.0,Jaws Collection


In [5]:
# Count how many dataset films belong to each TMDb collection.
collection_sizes = (
    films_analysis.loc[
        films_analysis["collection_id"].notna()
    ]
    .groupby(
        ["collection_id", "collection_name"],
        as_index=False
    )
    .agg(film_count=("id", "count"))
    .sort_values(
        ["film_count", "collection_name"],
        ascending=[False, True]
    )
)

# Display collection membership counts.
display(collection_sizes)

# Summarise how many collections have multiple films in this dataset.
multi_film_collections = (collection_sizes["film_count"] > 1).sum()
single_film_collections = (collection_sizes["film_count"] == 1).sum()

print(f"Collections with multiple films: {multi_film_collections}")
print(f"Collections with one film: {single_film_collections}")

,collection_id,collection_name,film_count
10,1241.0,Harry Potter Collection,8
0,10.0,Star Wars Collection,6
24,87359.0,Mission: Impossible Collection,5
6,295.0,Pirates of the Caribbean Collection,5
17,8091.0,Alien Collection,4
11,1570.0,Die Hard Collection,4
1,84.0,Indiana Jones Collection,4
12,2150.0,Shrek Collection,4
23,86311.0,The Avengers Collection,4
9,528.0,The Terminator Collection,4


Collections with multiple films: 23
Collections with one film: 9


In [6]:
# Check whether each TMDb movie ID is unique.
print("Unique movie IDs:", films_analysis["id"].is_unique)
print("Duplicate movie IDs:", films_analysis["id"].duplicated().sum())

# Parse release dates without modifying the original column.
films_analysis["release_date_parsed"] = pd.to_datetime(
    films_analysis["release_date"],
    errors="coerce"
)

# Identify missing or invalid release dates.
invalid_dates = films_analysis[
    films_analysis["release_date_parsed"].isna()
]

print("Missing or invalid release dates:", len(invalid_dates))

# Extract release year for historical analysis.
films_analysis["release_year"] = (
    films_analysis["release_date_parsed"].dt.year
)

# Display the release-year range.
print(
    "Earliest release year:",
    films_analysis["release_year"].min()
)
print(
    "Latest release year:",
    films_analysis["release_year"].max()
)

Unique movie IDs: True
Duplicate movie IDs: 0
Missing or invalid release dates: 0
Earliest release year: 1966
Latest release year: 2019


In [8]:
# Group films by historical release period.
period_summary = (
    films_analysis
    .groupby("release_period")
    .agg(
        film_count=("id", "count"),
        average_budget=("budget", "mean"),
        median_budget=("budget", "median"),
        average_revenue=("revenue", "mean"),
        median_revenue=("revenue", "median")
    )
)

# Format monetary figures with thousands separators and two decimals.
money_columns = [
    "average_budget",
    "median_budget",
    "average_revenue",
    "median_revenue"
]

formatted_summary = period_summary.copy()

for column in money_columns:
    formatted_summary[column] = formatted_summary[column].apply(
        lambda value: f"{value:,.2f}"
    )

# Display the financial summary.
display(formatted_summary)

,film_count,average_budget,median_budget,average_revenue,median_revenue
release_period,,,,,
2000 onward,71,"147,964,788.73","150,000,000.00","743,501,548.96","738,599,701.00"
Pre-2000,41,"40,010,975.61","30,000,000.00","350,565,165.54","321,365,567.00"


In [10]:
# Check missing and zero values in financial columns.
print("Missing budget values:", films_analysis["budget"].isna().sum())
print("Missing revenue values:", films_analysis["revenue"].isna().sum())

print("Zero budget values:", (films_analysis["budget"] == 0).sum())
print("Zero revenue values:", (films_analysis["revenue"] == 0).sum())

# Display minimum and maximum financial values.
print("\nFinancial ranges:")
display(
    films_analysis[["budget", "revenue"]].agg(["min", "max"])
)

# Display the 10 films with the highest reported budgets.
print("\nTop 10 films by reported budget:")
display(
    films_analysis.nlargest(10, "budget")[
        ["title", "release_year", "budget", "revenue"]
    ].style.format({
        "budget": "{:,.2f}",
        "revenue": "{:,.2f}"
    })
)

# Display the 10 films with the highest reported revenues.
print("\nTop 10 films by reported revenue:")
display(
    films_analysis.nlargest(10, "revenue")[
        ["title", "release_year", "budget", "revenue"]
    ].style.format({
        "budget": "{:,.2f}",
        "revenue": "{:,.2f}"
    })
)

Missing budget values: 0
Missing revenue values: 0
Zero budget values: 0
Zero revenue values: 0

Financial ranges:


,budget,revenue
min,1200000,7390108
max,379000000,2942649778



Top 10 films by reported budget:


,title,release_year,budget,revenue
28,Pirates of the Caribbean: On Stranger Tides,2011,"379,000,000.00","1,046,721,266.00"
54,Avengers: Endgame,2019,"356,000,000.00","2,942,649,778.00"
51,Pirates of the Caribbean: At World's End,2007,"300,000,000.00","961,691,209.00"
55,Avengers: Infinity War,2018,"300,000,000.00","2,052,415,039.00"
11,The Hobbit: The Battle of the Five Armies,2014,"250,000,000.00","956,019,788.00"
12,Harry Potter and the Deathly Hallows: Part 1,2010,"250,000,000.00","954,305,868.00"
66,The Dark Knight Rises,2012,"250,000,000.00","1,081,041,287.00"
67,The Hobbit: An Unexpected Journey,2012,"250,000,000.00","1,021,103,568.00"
72,The Hobbit: The Desolation of Smaug,2013,"250,000,000.00","958,400,000.00"
90,Harry Potter and the Half-Blood Prince,2009,"250,000,000.00","933,959,197.00"



Top 10 films by reported revenue:


,title,release_year,budget,revenue
54,Avengers: Endgame,2019,"356,000,000.00","2,942,649,778.00"
55,Avengers: Infinity War,2018,"300,000,000.00","2,052,415,039.00"
42,The Avengers,2012,"220,000,000.00","1,518,815,515.00"
111,Avengers: Age of Ultron,2015,"235,000,000.00","1,405,403,694.00"
13,Harry Potter and the Deathly Hallows: Part 2,2011,"125,000,000.00","1,341,511,219.00"
88,Iron Man 3,2013,"200,000,000.00","1,215,577,205.00"
10,The Lord of the Rings: The Return of the King,2003,"94,000,000.00","1,118,888,979.00"
66,The Dark Knight Rises,2012,"250,000,000.00","1,081,041,287.00"
56,Toy Story 4,2019,"175,000,000.00","1,073,841,394.00"
2,Toy Story 3,2010,"200,000,000.00","1,067,316,101.00"


In [11]:
# Summarise audience indicators across all films.
audience_summary = films_analysis[
    ["popularity", "vote_average", "vote_count"]
].agg(
    ["count", "mean", "median", "min", "max"]
).round(2)

display(audience_summary)

# Identify the 10 highest-rated films.
print("Top 10 films by average rating:")

display(
    films_analysis.nlargest(10, "vote_average")[
        ["title", "release_year", "vote_average", "vote_count", "popularity"]
    ].style.format({
        "vote_average": "{:.2f}",
        "vote_count": "{:,.0f}",
        "popularity": "{:,.2f}"
    })
)

# Identify the 10 most popular films according to TMDb.
print("Top 10 films by TMDb popularity:")

display(
    films_analysis.nlargest(10, "popularity")[
        ["title", "release_year", "popularity", "vote_average", "vote_count"]
    ].style.format({
        "popularity": "{:,.2f}",
        "vote_average": "{:.2f}",
        "vote_count": "{:,.0f}"
    })
)

,popularity,vote_average,vote_count
count,112.00,112.00,112.00
mean,19.75,7.59,17548.12
median,15.79,7.72,15850.50
min,0.46,5.75,4137.00
max,85.70,8.73,41950.00


Top 10 films by average rating:


,title,release_year,vote_average,vote_count,popularity
49,The Shawshank Redemption,1994,8.73,"31,879",45.41
38,The Godfather,1972,8.69,"24,013",42.42
40,The Godfather Part II,1974,8.57,"14,638",25.76
63,Schindler's List,1993,8.57,"18,205",22.81
14,Spirited Away,2001,8.54,"19,245",49.67
10,The Lord of the Rings: The Return of the King,2003,8.51,"27,862",34.24
8,The Lord of the Rings: The Fellowship of the Ring,2001,8.50,"28,905",39.99
17,The Dark Knight,2008,8.50,"37,425",52.52
21,Interstellar,2014,8.50,"41,950",72.25
87,Pulp Fiction,1994,8.48,"31,365",30.00


Top 10 films by TMDb popularity:


,title,release_year,popularity,vote_average,vote_count
54,Avengers: Endgame,2019,85.70,8.20,"29,432"
21,Interstellar,2014,72.25,8.50,"41,950"
17,The Dark Knight,2008,52.52,8.50,"37,425"
47,Inception,2010,49.75,8.37,"40,769"
14,Spirited Away,2001,49.67,8.54,"19,245"
55,Avengers: Infinity War,2018,47.28,8.25,"33,433"
49,The Shawshank Redemption,1994,45.41,8.73,"31,879"
38,The Godfather,1972,42.42,8.69,"24,013"
81,Harry Potter and the Philosopher's Stone,2001,41.37,7.91,"30,653"
42,The Avengers,2012,41.32,8.09,"40,407"


In [12]:
from pathlib import Path
import json

# Locate the cached movie credits directory.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "explore" else Path.cwd()
CREDITS_DIR = PROJECT_ROOT / "data" / "cache" / "credits"

# Load successful credits responses.
credit_records = []

for file_path in sorted(CREDITS_DIR.glob("*.json")):
    with file_path.open("r", encoding="utf-8") as file:
        response = json.load(file)

    credit_records.append(response["data"])

# Display the number of loaded credit records.
print("Total credit records:", len(credit_records))

# Inspect the structure of one credits response.
sample_credit = credit_records[0]

print("\nTop-level fields:")
print(sample_credit.keys())

print("\nSample cast record:")
display(sample_credit["cast"][:3])

print("\nSample crew record:")
display(sample_credit["crew"][:3])

Total credit records: 112

Top-level fields:
dict_keys(['id', 'cast', 'crew'])

Sample cast record:


[{'adult': False,
  'gender': 2,
  'id': 3223,
  'known_for_department': 'Acting',
  'name': 'Robert Downey Jr.',
  'original_name': 'Robert Downey Jr.',
  'popularity': 10.2234,
  'profile_path': '/5qHNjhtjMD4YWH3UP0rm4tKwxCL.jpg',
  'cast_id': 30,
  'character': 'Tony Stark',
  'credit_id': '52fe43329251416c75007341',
  'order': 0},
 {'adult': False,
  'gender': 1,
  'id': 12052,
  'known_for_department': 'Acting',
  'name': 'Gwyneth Paltrow',
  'original_name': 'Gwyneth Paltrow',
  'popularity': 2.8033,
  'profile_path': '/r4wbWWEEjmtRFZ0GU10XbLTgp47.jpg',
  'cast_id': 18,
  'character': 'Pepper Potts',
  'credit_id': '52fe43329251416c75007307',
  'order': 1},
 {'adult': False,
  'gender': 2,
  'id': 1896,
  'known_for_department': 'Acting',
  'name': 'Don Cheadle',
  'original_name': 'Don Cheadle',
  'popularity': 2.4944,
  'profile_path': '/umhTmcVF26qZKEpSfG1ZNpcbs9D.jpg',
  'cast_id': 19,
  'character': "Lt. Col. James 'Rhodey' Rhodes",
  'credit_id': '52fe43329251416c7500730b',


Sample crew record:


[{'adult': False,
  'gender': 2,
  'id': 1463281,
  'known_for_department': 'Directing',
  'name': 'Chris Castaldi',
  'original_name': 'Chris Castaldi',
  'popularity': 0.2821,
  'profile_path': '/8wndmlAfA7R5JuGwxTK7S4TB64O.jpg',
  'credit_id': '604a96a990fca3006f797d55',
  'department': 'Directing',
  'job': 'Second Assistant Director'},
 {'adult': False,
  'gender': 0,
  'id': 1579412,
  'known_for_department': 'Visual Effects',
  'name': 'Wayne Billheimer',
  'original_name': 'Wayne Billheimer',
  'popularity': 0.2768,
  'profile_path': None,
  'credit_id': '604a96f99e458600792f48b0',
  'department': 'Visual Effects',
  'job': 'Visual Effects Producer'},
 {'adult': False,
  'gender': 2,
  'id': 24192,
  'known_for_department': 'Sound',
  'name': 'Dave Jordan',
  'original_name': 'Dave Jordan',
  'popularity': 0.4155,
  'profile_path': '/btzUgkKCllxClxIgLhmhZ1qaPw.jpg',
  'credit_id': '604a94f990fca3005a8b3709',
  'department': 'Sound',
  'job': 'Music Supervisor'}]

In [13]:
from collections import Counter

# Calculate cast and crew sizes for each film.
cast_sizes = [len(record["cast"]) for record in credit_records]
crew_sizes = [len(record["crew"]) for record in credit_records]

print("CAST PROFILE")
print("Total cast credits:", sum(cast_sizes))
print("Average cast size:", round(sum(cast_sizes) / len(cast_sizes), 2))
print("Smallest cast:", min(cast_sizes))
print("Largest cast:", max(cast_sizes))

print("\nCREW PROFILE")
print("Total crew credits:", sum(crew_sizes))
print("Average crew size:", round(sum(crew_sizes) / len(crew_sizes), 2))
print("Smallest crew:", min(crew_sizes))
print("Largest crew:", max(crew_sizes))

# Identify available crew departments.
departments = Counter(
    member.get("department")
    for record in credit_records
    for member in record["crew"]
)

print("\nCREW DEPARTMENTS")
display(
    pd.DataFrame(
        departments.items(),
        columns=["department", "credit_count"]
    ).sort_values("credit_count", ascending=False)
)

# Identify available crew job titles.
jobs = Counter(
    member.get("job")
    for record in credit_records
    for member in record["crew"]
)

print("\nTOP 20 CREW JOB TITLES")
display(
    pd.DataFrame(
        jobs.most_common(20),
        columns=["job", "credit_count"]
    )
)

CAST PROFILE
Total cast credits: 7827
Average cast size: 69.88
Smallest cast: 9
Largest cast: 177

CREW PROFILE
Total crew credits: 24932
Average crew size: 222.61
Smallest crew: 40
Largest crew: 953

CREW DEPARTMENTS


,department,credit_count
9,Crew,6191
1,Visual Effects,3554
2,Sound,3129
3,Production,2833
6,Art,2558
7,Costume & Make-Up,1902
8,Camera,1515
0,Directing,999
10,Lighting,947
5,Editing,762



TOP 20 CREW JOB TITLES


,job,credit_count
0,Stunts,2218
1,Animation,624
2,Makeup Artist,381
3,Stunt Double,339
4,Visual Effects Supervisor,324
5,Art Direction,322
6,Musician,322
7,Executive Producer,312
8,Visual Effects Coordinator,272
9,Sound Effects Editor,243


In [14]:
from collections import Counter

# Identify the most common cast billing positions.
cast_orders = Counter(
    member.get("order")
    for record in credit_records
    for member in record["cast"]
)

print("Most common cast billing positions:")
display(
    pd.DataFrame(
        cast_orders.most_common(10),
        columns=["billing_order", "credit_count"]
    )
)

# Inspect relevant creative crew roles.
relevant_jobs = {
    "Director",
    "Writer",
    "Screenplay",
    "Story",
    "Producer",
    "Executive Producer"
}

creative_credits = [
    member
    for record in credit_records
    for member in record["crew"]
    if member.get("job") in relevant_jobs
]

# Summarise relevant creative roles.
creative_role_counts = Counter(
    member.get("job")
    for member in creative_credits
)

print("\nRelevant creative roles:")
display(
    pd.DataFrame(
        creative_role_counts.items(),
        columns=["job", "credit_count"]
    ).sort_values("credit_count", ascending=False)
)

Most common cast billing positions:


,billing_order,credit_count
0,0,112
1,1,112
2,2,112
3,3,112
4,4,112
5,5,112
6,6,112
7,7,112
8,8,112
9,9,111



Relevant creative roles:


,job,credit_count
0,Executive Producer,312
2,Producer,238
1,Screenplay,169
3,Director,122
5,Story,71
4,Writer,45


In [15]:
# Display all relevant creative role counts, including missing job values.
creative_role_summary = (
    pd.Series(
        [
            member.get("job")
            for record in credit_records
            for member in record["crew"]
            if member.get("job") in relevant_jobs
        ],
        name="job"
    )
    .value_counts(dropna=False)
    .rename_axis("job")
    .reset_index(name="credit_count")
)

display(creative_role_summary)

,job,credit_count
0,Executive Producer,312
1,Producer,238
2,Screenplay,169
3,Director,122
4,Story,71
5,Writer,45


In [16]:
# Prepare focused cast and crew records.
cast_rows = []
director_rows = []
writer_rows = []
producer_rows = []

for record in credit_records:
    movie_id = record["id"]

    # Select the first five billed cast members.
    for member in record["cast"]:
        if member.get("order") is not None and member["order"] <= 4:
            cast_rows.append({
                "movie_id": movie_id,
                "person_id": member.get("id"),
                "person_name": member.get("name"),
                "character": member.get("character"),
                "billing_order": member.get("order"),
                "person_popularity": member.get("popularity")
            })

    # Extract relevant creative crew roles.
    for member in record["crew"]:
        job = member.get("job")

        row = {
            "movie_id": movie_id,
            "person_id": member.get("id"),
            "person_name": member.get("name"),
            "department": member.get("department"),
            "job": job,
            "person_popularity": member.get("popularity")
        }

        if job == "Director":
            director_rows.append(row)

        elif job in {"Writer", "Screenplay", "Story"}:
            writer_rows.append(row)

        elif job in {"Producer", "Executive Producer"}:
            producer_rows.append(row)

# Convert extracted records into analytical DataFrames.
film_cast = pd.DataFrame(cast_rows)
film_directors = pd.DataFrame(director_rows)
film_writers = pd.DataFrame(writer_rows)
film_producers = pd.DataFrame(producer_rows)

# Display dataset sizes.
print("Lead cast records:", len(film_cast))
print("Director records:", len(film_directors))
print("Writer records:", len(film_writers))
print("Producer records:", len(film_producers))

Lead cast records: 560
Director records: 122
Writer records: 285
Producer records: 550


In [17]:
# Validate the four talent datasets.
talent_datasets = {
    "Lead cast": film_cast,
    "Directors": film_directors,
    "Writers": film_writers,
    "Producers": film_producers
}

for dataset_name, dataset in talent_datasets.items():
    print(f"\n{dataset_name}")
    print("-" * 40)

    print("Records:", len(dataset))
    print("Missing person IDs:", dataset["person_id"].isna().sum())
    print("Missing names:", dataset["person_name"].isna().sum())

    # Check duplicate person-film-role combinations.
    duplicate_count = dataset.duplicated(
        subset=["movie_id", "person_id", "job"]
        if "job" in dataset.columns
        else ["movie_id", "person_id"]
    ).sum()

    print("Duplicate credit combinations:", duplicate_count)

# Verify lead cast coverage.
print("\nLead cast film coverage:")
print("Films represented:", film_cast["movie_id"].nunique())
print("Expected films:", films_analysis["id"].nunique())

# Identify people appearing in multiple films.
cast_person_counts = (
    film_cast.groupby(["person_id", "person_name"])["movie_id"]
    .nunique()
    .reset_index(name="film_count")
    .sort_values("film_count", ascending=False)
)

print("\nPeople appearing in the most films:")
display(cast_person_counts.head(10))


Lead cast
----------------------------------------
Records: 560
Missing person IDs: 0
Missing names: 0
Duplicate credit combinations: 0

Directors
----------------------------------------
Records: 122
Missing person IDs: 0
Missing names: 0
Duplicate credit combinations: 0

Writers
----------------------------------------
Records: 285
Missing person IDs: 0
Missing names: 0
Duplicate credit combinations: 0

Producers
----------------------------------------
Records: 550
Missing person IDs: 0
Missing names: 0
Duplicate credit combinations: 0

Lead cast film coverage:
Films represented: 112
Expected films: 112

People appearing in the most films:


,person_id,person_name,film_count
1,3,Harrison Ford,8
239,10990,Emma Watson,8
237,10980,Daniel Radcliffe,8
238,10989,Rupert Grint,8
104,1892,Matt Damon,7
139,3223,Robert Downey Jr.,7
91,1327,Ian McKellen,6
233,10205,Sigourney Weaver,5
13,85,Johnny Depp,5
146,3894,Christian Bale,5


In [18]:
# Summarise talent appearances and popularity across films.
def build_talent_summary(dataset, include_billing=False):
    aggregation = {
        "film_count": ("movie_id", "nunique"),
        "average_popularity": ("person_popularity", "mean"),
        "median_popularity": ("person_popularity", "median"),
        "maximum_popularity": ("person_popularity", "max")
    }

    # Include cast billing information for actors only.
    if include_billing:
        aggregation["average_billing_order"] = (
            "billing_order", "mean"
        )

    summary = (
        dataset.groupby(
            ["person_id", "person_name"],
            as_index=False
        )
        .agg(**aggregation)
        .round(2)
        .sort_values(
            ["film_count", "average_popularity"],
            ascending=[False, False]
        )
    )

    return summary


# Build separate summaries for each talent category.
cast_talent_summary = build_talent_summary(
    film_cast,
    include_billing=True
)

director_talent_summary = build_talent_summary(film_directors)
writer_talent_summary = build_talent_summary(film_writers)
producer_talent_summary = build_talent_summary(film_producers)


# Inspect the leading recurring talent in each category.
print("TOP 15 ACTORS")
display(cast_talent_summary.head(15))

print("TOP 10 DIRECTORS")
display(director_talent_summary.head(10))

print("TOP 10 WRITERS")
display(writer_talent_summary.head(10))

print("TOP 10 PRODUCERS")
display(producer_talent_summary.head(10))

TOP 15 ACTORS


,person_id,person_name,film_count,average_popularity,median_popularity,maximum_popularity,average_billing_order
239,10990,Emma Watson,8,6.56,6.56,6.56,1.75
1,3,Harrison Ford,8,4.44,4.44,4.44,0.38
237,10980,Daniel Radcliffe,8,4.25,4.25,4.25,0.00
238,10989,Rupert Grint,8,4.00,4.00,4.00,1.25
139,3223,Robert Downey Jr.,7,10.22,10.22,10.22,0.00
104,1892,Matt Damon,7,8.33,8.33,8.33,1.43
91,1327,Ian McKellen,6,4.49,4.49,4.49,0.67
13,85,Johnny Depp,5,9.31,9.31,9.31,0.00
5,31,Tom Hanks,5,9.00,9.00,9.00,0.00
34,500,Tom Cruise,5,8.77,8.77,8.77,0.00


TOP 10 DIRECTORS


,person_id,person_name,film_count,average_popularity,median_popularity,maximum_popularity
6,488,Steven Spielberg,8,4.96,4.96,4.96
7,525,Christopher Nolan,7,8.27,8.27,8.27
4,108,Peter Jackson,6,2.49,2.49,2.49
0,1,George Lucas,4,2.37,2.37,2.37
3,24,Robert Zemeckis,4,2.08,2.08,2.08
40,11343,David Yates,4,1.22,1.22,1.22
5,138,Quentin Tarantino,3,4.94,4.94,4.94
19,2710,James Cameron,3,3.46,3.46,3.46
16,1776,Francis Ford Coppola,3,1.89,1.89,1.89
17,1884,Steven Soderbergh,3,1.64,1.64,1.64


TOP 10 WRITERS


,person_id,person_name,film_count,average_popularity,median_popularity,maximum_popularity
0,1,George Lucas,10,2.37,2.37,2.37
10,525,Christopher Nolan,7,8.27,8.27,8.27
86,10967,Steve Kloves,7,0.67,0.67,0.67
5,108,Peter Jackson,6,2.49,2.49,2.49
1,7,Andrew Stanton,6,1.64,1.64,1.64
6,126,Fran Walsh,6,1.01,1.01,1.01
7,128,Philippa Boyens,6,0.97,0.97,0.97
30,1706,Terry Rossio,6,0.63,0.63,0.63
29,1705,Ted Elliott,5,0.66,0.66,0.66
11,527,Jonathan Nolan,4,1.73,1.73,1.73


TOP 10 PRODUCERS


,person_id,person_name,film_count,average_popularity,median_popularity,maximum_popularity
0,1,George Lucas,10,2.37,2.37,2.37
25,664,Frank Marshall,10,0.66,0.66,0.66
124,10968,David Heyman,8,1.12,1.12,1.12
14,489,Kathleen Kennedy,8,0.64,0.64,0.64
157,15277,Jon Favreau,7,2.94,2.94,2.94
91,7624,Stan Lee,7,2.74,2.74,2.74
117,10850,Kevin Feige,7,2.56,2.56,2.56
212,57027,Louis D'Esposito,7,1.00,1.00,1.00
39,1296,Bruce Berman,7,0.29,0.29,0.29
13,488,Steven Spielberg,6,4.96,4.96,4.96


In [19]:
# Check whether each person has one consistent popularity value.
popularity_checks = {}

for dataset_name, dataset in talent_datasets.items():
    person_popularity_counts = (
        dataset.groupby("person_id")["person_popularity"]
        .nunique(dropna=True)
    )

    popularity_checks[dataset_name] = {
        "people": len(person_popularity_counts),
        "people_with_multiple_popularity_values": (
            person_popularity_counts > 1
        ).sum()
    }

display(pd.DataFrame(popularity_checks).T)

,people,people_with_multiple_popularity_values
Lead cast,330,0
Directors,66,0
Writers,160,0
Producers,277,0


In [20]:
# Extract individual genre records from each film.
genre_rows = []

for _, film in films_analysis.iterrows():
    for genre in films_df.loc[
        films_df["id"] == film["id"], "genres"
    ].iloc[0]:
        genre_rows.append({
            "movie_id": film["id"],
            "title": film["title"],
            "release_year": film["release_year"],
            "genre_id": genre.get("id"),
            "genre_name": genre.get("name"),
            "budget": film["budget"],
            "revenue": film["revenue"],
            "popularity": film["popularity"],
            "vote_average": film["vote_average"],
            "vote_count": film["vote_count"]
        })

# Create a film-genre analytical dataset.
film_genres = pd.DataFrame(genre_rows)

# Inspect the extracted records.
print("Total film-genre records:", len(film_genres))
print("Unique genres:", film_genres["genre_name"].nunique())

# Display genre distribution.
genre_distribution = (
    film_genres.groupby("genre_name")
    .agg(
        film_count=("movie_id", "nunique")
    )
    .sort_values("film_count", ascending=False)
)

display(genre_distribution)

Total film-genre records: 320
Unique genres: 16


,film_count
genre_name,
Action,61
Adventure,60
Science Fiction,36
Thriller,33
Fantasy,27
Comedy,19
Drama,19
Crime,18
Family,17


In [21]:
# Compare audience and financial indicators across genres.
genre_performance = (
    film_genres.groupby("genre_name")
    .agg(
        film_count=("movie_id", "nunique"),
        average_popularity=("popularity", "mean"),
        average_rating=("vote_average", "mean"),
        average_vote_count=("vote_count", "mean"),
        median_revenue=("revenue", "median"),
        median_budget=("budget", "median")
    )
    .round(2)
    .sort_values("film_count", ascending=False)
)

# Format financial figures for readability.
for column in ["median_revenue", "median_budget"]:
    genre_performance[column] = genre_performance[column].apply(
        lambda value: f"{value:,.2f}"
    )

display(genre_performance)

,film_count,average_popularity,average_rating,average_vote_count,median_revenue,median_budget
genre_name,,,,,,
Action,61,18.66,7.41,17162.41,"585,174,222.00","115,000,000.00"
Adventure,60,20.31,7.44,18439.28,"781,017,020.00","125,000,000.00"
Science Fiction,36,21.77,7.52,19472.22,"529,855,930.00","107,500,000.00"
Thriller,33,14.17,7.39,13562.76,"366,101,666.00","85,000,000.00"
Fantasy,27,22.71,7.54,19024.63,"926,287,400.00","150,000,000.00"
Comedy,19,16.11,7.44,15644.68,"563,749,323.00","80,000,000.00"
Drama,19,23.59,8.18,19670.05,"245,066,411.00","40,000,000.00"
Crime,18,23.12,8.00,20269.33,"258,904,666.50","42,000,000.00"
Family,17,17.60,7.41,15052.29,"650,000,000.00","92,000,000.00"


In [22]:
# Compare the films assigned to Family and Animation.
family_movies = set(
    film_genres.loc[
        film_genres["genre_name"] == "Family",
        "movie_id"
    ]
)

animation_movies = set(
    film_genres.loc[
        film_genres["genre_name"] == "Animation",
        "movie_id"
    ]
)

print("Family films:", len(family_movies))
print("Animation films:", len(animation_movies))
print("Same movie membership:", family_movies == animation_movies)

# Display films belonging to only one of the two genres.
print("\nFamily only:")
display(
    film_genres[
        (film_genres["genre_name"] == "Family") &
        (~film_genres["movie_id"].isin(animation_movies))
    ][["movie_id", "title"]].drop_duplicates()
)

print("\nAnimation only:")
display(
    film_genres[
        (film_genres["genre_name"] == "Animation") &
        (~film_genres["movie_id"].isin(family_movies))
    ][["movie_id", "title"]].drop_duplicates()
)

# Display the genre summary with clearly labelled numeric columns.
display(
    film_genres.groupby("genre_name").agg(
        film_count=("movie_id", "nunique"),
        average_popularity=("popularity", "mean"),
        average_rating=("vote_average", "mean"),
        average_vote_count=("vote_count", "mean"),
        median_revenue=("revenue", "median"),
        median_budget=("budget", "median")
    ).round(2)
)

Family films: 17
Animation films: 17
Same movie membership: True

Family only:


,movie_id,title



Animation only:


,movie_id,title


,film_count,average_popularity,average_rating,average_vote_count,median_revenue,median_budget
genre_name,,,,,,
Action,61,18.66,7.41,17162.41,585174222.0,115000000.0
Adventure,60,20.31,7.44,18439.28,781017020.0,125000000.0
Animation,17,17.60,7.41,15052.29,650000000.0,92000000.0
Comedy,19,16.11,7.44,15644.68,563749323.0,80000000.0
Crime,18,23.12,8.00,20269.33,258904666.5,42000000.0
Drama,19,23.59,8.18,19670.05,245066411.0,40000000.0
Family,17,17.60,7.41,15052.29,650000000.0,92000000.0
Fantasy,27,22.71,7.54,19024.63,926287400.0,150000000.0
History,1,22.81,8.57,18205.00,321365567.0,22000000.0


In [26]:
# Summarise films using the original collection columns.
franchise_summary = (
    franchise_analysis.dropna(subset=["collection_id_x"])
    .groupby(
        ["collection_id_x", "collection_name_x"],
        as_index=False
    )
    .agg(
        film_count=("id", "nunique"),
        average_rating=("vote_average", "mean"),
        average_popularity=("popularity", "mean"),
        average_vote_count=("vote_count", "mean"),
        median_budget=("budget", "median"),
        median_revenue=("revenue", "median"),
        total_reported_revenue=("revenue", "sum")
    )
    .sort_values(
        ["film_count", "average_popularity"],
        ascending=[False, False]
    )
)

display(franchise_summary.head(15).round(2))

,collection_id_x,collection_name_x,film_count,average_rating,average_popularity,average_vote_count,median_budget,median_revenue,total_reported_revenue
10,1241.0,Harry Potter Collection,8,7.82,28.19,23606.75,140000000.0,9.360860e+08,7759777709
0,10.0,Star Wars Collection,6,7.53,13.83,17798.50,72675000.0,7.123982e+08,4432411744
6,295.0,Pirates of the Caribbean Collection,5,7.15,16.64,17153.20,230000000.0,9.616912e+08,4525525744
24,87359.0,Mission: Impossible Collection,5,6.86,9.28,9550.60,145000000.0,5.463881e+08,2779994012
23,86311.0,The Avengers Collection,4,7.95,51.71,32161.25,267500000.0,1.785615e+09,7919284026
21,10194.0,Toy Story Collection,4,7.72,17.25,16019.00,132500000.0,7.823458e+08,3039690868
12,2150.0,Shrek Collection,4,6.96,14.90,12996.00,155000000.0,7.804555e+08,2983130036
9,528.0,The Terminator Collection,4,7.05,14.60,11418.75,151000000.0,4.031765e+08,1402538376
17,8091.0,Alien Collection,4,7.18,13.46,10293.75,34250000.0,1.608868e+08,610021801
1,84.0,Indiana Jones Collection,4,7.28,11.46,11430.75,38000000.0,4.320489e+08,1983841081


In [25]:
# Inspect the columns currently available.
print(franchise_analysis.columns.tolist())


['id', 'title', 'release_date', 'belongs_to_collection', 'budget', 'revenue', 'popularity', 'vote_average', 'vote_count', 'collection_id_x', 'collection_name_x', 'release_date_parsed', 'release_year', 'release_period', 'movie_id', 'collection_id_y', 'collection_name_y']


In [27]:
# Display the franchise summary with explicit, readable formatting.
franchise_display = franchise_summary.copy()

numeric_columns = [
    "average_rating",
    "average_popularity",
    "average_vote_count"
]

for column in numeric_columns:
    franchise_display[column] = franchise_display[column].round(2)

currency_columns = [
    "median_budget",
    "median_revenue",
    "total_reported_revenue"
]

for column in currency_columns:
    franchise_display[column] = franchise_display[column].map(
        lambda value: f"{value:,.0f}"
    )

display(franchise_display.head(15))

# Inspect films that do not have a TMDb collection object.
non_collection_films = franchise_analysis[
    franchise_analysis["collection_id_x"].isna()
][[
    "id",
    "title",
    "release_year",
    "vote_average",
    "popularity",
    "budget",
    "revenue"
]].sort_values("popularity", ascending=False)

print("Films without a TMDb collection:", len(non_collection_films))
display(non_collection_films)

,collection_id_x,collection_name_x,film_count,average_rating,average_popularity,average_vote_count,median_budget,median_revenue,total_reported_revenue
10,1241.0,Harry Potter Collection,8,7.82,28.19,23606.75,"140,000,000","936,085,968","7,759,777,709"
0,10.0,Star Wars Collection,6,7.53,13.83,17798.50,"72,675,000","712,398,168","4,432,411,744"
6,295.0,Pirates of the Caribbean Collection,5,7.15,16.64,17153.20,"230,000,000","961,691,209","4,525,525,744"
24,87359.0,Mission: Impossible Collection,5,6.86,9.28,9550.60,"145,000,000","546,388,108","2,779,994,012"
23,86311.0,The Avengers Collection,4,7.95,51.71,32161.25,"267,500,000","1,785,615,277","7,919,284,026"
21,10194.0,Toy Story Collection,4,7.72,17.25,16019.00,"132,500,000","782,345,752","3,039,690,868"
12,2150.0,Shrek Collection,4,6.96,14.90,12996.00,"155,000,000","780,455,527","2,983,130,036"
9,528.0,The Terminator Collection,4,7.05,14.60,11418.75,"151,000,000","403,176,500","1,402,538,376"
17,8091.0,Alien Collection,4,7.18,13.46,10293.75,"34,250,000","160,886,772","610,021,801"
1,84.0,Indiana Jones Collection,4,7.28,11.46,11430.75,"38,000,000","432,048,888","1,983,841,081"


Films without a TMDb collection: 16


,id,title,release_year,vote_average,popularity,budget,revenue
21,157336,Interstellar,2014,8.500,72.2480,165000000,746606706
47,27205,Inception,2010,8.373,49.7497,160000000,839030630
14,129,Spirited Away,2001,8.537,49.6655,19000000,274925095
49,278,The Shawshank Redemption,1994,8.734,45.4140,25000000,28341469
69,550,Fight Club,1999,8.437,36.9333,63000000,100853753
87,680,Pulp Fiction,1994,8.480,29.9996,8000000,213928762
15,13,Forrest Gump,1994,8.462,29.2116,55000000,677387716
63,424,Schindler's List,1993,8.570,22.8137,22000000,321365567
91,769,GoodFellas,1990,8.452,20.3111,25000000,47072327
4,10681,WALL·E,2008,8.129,19.0410,180000000,521311860


In [28]:
# Add a period classification to films without a TMDb collection.
non_collection_films = non_collection_films.copy()

non_collection_films["release_period"] = non_collection_films[
    "release_year"
].apply(
    lambda year: "Pre-2000" if year < 2000 else "2000 onward"
)

# Compare the number of potential standalone candidates by period.
print("Potential standalone candidates by release period:")
display(
    non_collection_films.groupby("release_period").agg(
        film_count=("id", "nunique"),
        average_rating=("vote_average", "mean"),
        average_popularity=("popularity", "mean"),
        median_revenue=("revenue", "median"),
        median_budget=("budget", "median")
    ).round(2)
)

# Display the 2000-onward candidates first.
print("\nPotential standalone candidates released from 2000 onward:")
display(
    non_collection_films[
        non_collection_films["release_year"] >= 2000
    ].sort_values(
        ["popularity", "vote_average"],
        ascending=False
    )
)

Potential standalone candidates by release period:


,film_count,average_rating,average_popularity,median_revenue,median_budget
release_period,,,,,
2000 onward,8,8.26,31.01,283195047.5,65000000.0
Pre-2000,8,8.47,27.70,83438541.0,25000000.0



Potential standalone candidates released from 2000 onward:


,id,title,release_year,vote_average,popularity,budget,revenue,release_period
21,157336,Interstellar,2014,8.500,72.2480,165000000,746606706,2000 onward
47,27205,Inception,2010,8.373,49.7497,160000000,839030630,2000 onward
14,129,Spirited Away,2001,8.537,49.6655,19000000,274925095,2000 onward
4,10681,WALL·E,2008,8.129,19.0410,180000000,521311860,2000 onward
16,1422,The Departed,2006,8.160,18.7587,90000000,291465000,2000 onward
7,1124,The Prestige,2006,8.212,18.4127,40000000,109676311,2000 onward
92,77,Memento,2000,8.175,14.8529,9000000,40060108,2000 onward
80,641,Requiem for a Dream,2000,8.007,5.3724,4500000,7390108,2000 onward


In [30]:
# Rebuild the recent candidate dataset with all required film metrics.
non_collection_films = franchise_analysis[
    franchise_analysis["collection_id_x"].isna()
][[
    "id",
    "title",
    "release_year",
    "vote_average",
    "vote_count",
    "popularity",
    "budget",
    "revenue"
]].sort_values("popularity", ascending=False)

# Select candidates released from 2000 onward.
recent_candidates = non_collection_films[
    non_collection_films["release_year"] >= 2000
].copy()

print("Recent candidate count:", len(recent_candidates))
display(recent_candidates)

Recent candidate count: 8


,id,title,release_year,vote_average,vote_count,popularity,budget,revenue
21,157336,Interstellar,2014,8.500,41950,72.2480,165000000,746606706
47,27205,Inception,2010,8.373,40769,49.7497,160000000,839030630
14,129,Spirited Away,2001,8.537,19245,49.6655,19000000,274925095
4,10681,WALL·E,2008,8.129,21213,19.0410,180000000,521311860
16,1422,The Departed,2006,8.160,16955,18.7587,90000000,291465000
7,1124,The Prestige,2006,8.212,18487,18.4127,40000000,109676311
92,77,Memento,2000,8.175,17065,14.8529,9000000,40060108
80,641,Requiem for a Dream,2000,8.007,11500,5.3724,4500000,7390108


In [31]:
# Identify the eight recent candidate movie IDs.
candidate_ids = set(recent_candidates["id"])

# Aggregate genres for each candidate.
candidate_genres = (
    film_genres[film_genres["movie_id"].isin(candidate_ids)]
    .groupby("movie_id")["genre_name"]
    .agg(lambda values: ", ".join(sorted(set(values))))
    .rename("genres")
)

# Aggregate the five highest-billed cast members.
candidate_cast = (
    film_cast[
        (film_cast["movie_id"].isin(candidate_ids)) &
        (film_cast["billing_order"] <= 4)
    ]
    .sort_values(["movie_id", "billing_order"])
    .groupby("movie_id")["person_name"]
    .agg(lambda values: ", ".join(values))
    .rename("lead_cast")
)

# Aggregate directors for each candidate.
candidate_directors = (
    film_directors[
        film_directors["movie_id"].isin(candidate_ids)
    ]
    .groupby("movie_id")["person_name"]
    .agg(lambda values: ", ".join(sorted(set(values))))
    .rename("directors")
)

# Combine all candidate attributes into one DataFrame.
candidate_profiles = (
    recent_candidates.set_index("id")
    .join(candidate_genres)
    .join(candidate_cast)
    .join(candidate_directors)
    .reset_index()
)

# Display the consolidated candidate profiles.
display(
    candidate_profiles[
        [
            "id",
            "title",
            "release_year",
            "genres",
            "vote_average",
            "vote_count",
            "popularity",
            "lead_cast",
            "directors"
        ]
    ].sort_values("popularity", ascending=False)
)

,id,title,release_year,genres,vote_average,vote_count,popularity,lead_cast,directors
0,157336,Interstellar,2014,"Adventure, Drama, Science Fiction",8.500,41950,72.2480,"Matthew McConaughey, Anne Hathaway, Michael Ca...",Christopher Nolan
1,27205,Inception,2010,"Action, Crime, Science Fiction",8.373,40769,49.7497,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W...",Christopher Nolan
2,129,Spirited Away,2001,"Animation, Family, Fantasy",8.537,19245,49.6655,"Rumi Hiiragi, Miyu Irino, Mari Natsuki, Takash...",Hayao Miyazaki
3,10681,WALL·E,2008,"Animation, Family, Science Fiction",8.129,21213,19.0410,"Ben Burtt, Elissa Knight, Jeff Garlin, Fred Wi...",Andrew Stanton
4,1422,The Departed,2006,"Crime, Drama, Thriller",8.160,16955,18.7587,"Leonardo DiCaprio, Matt Damon, Jack Nicholson,...",Martin Scorsese
5,1124,The Prestige,2006,"Drama, Mystery, Science Fiction",8.212,18487,18.4127,"Hugh Jackman, Christian Bale, Michael Caine, P...",Christopher Nolan
6,77,Memento,2000,"Mystery, Thriller",8.175,17065,14.8529,"Guy Pearce, Carrie-Anne Moss, Joe Pantoliano, ...",Christopher Nolan
7,641,Requiem for a Dream,2000,Drama,8.007,11500,5.3724,"Ellen Burstyn, Jared Leto, Jennifer Connelly, ...",Darren Aronofsky


In [32]:
# Rank candidates independently on each audience indicator.
candidate_rankings = candidate_profiles[
    ["id", "title", "vote_average", "vote_count", "popularity"]
].copy()

candidate_rankings["rating_rank"] = (
    candidate_rankings["vote_average"]
    .rank(method="min", ascending=False)
    .astype(int)
)

candidate_rankings["vote_count_rank"] = (
    candidate_rankings["vote_count"]
    .rank(method="min", ascending=False)
    .astype(int)
)

candidate_rankings["popularity_rank"] = (
    candidate_rankings["popularity"]
    .rank(method="min", ascending=False)
    .astype(int)
)

# Display individual rankings without combining them into an arbitrary score.
candidate_rankings = candidate_rankings.sort_values("popularity_rank")

display(candidate_rankings)

,id,title,vote_average,vote_count,popularity,rating_rank,vote_count_rank,popularity_rank
0,157336,Interstellar,8.500,41950,72.2480,2,1,1
1,27205,Inception,8.373,40769,49.7497,3,2,2
2,129,Spirited Away,8.537,19245,49.6655,1,4,3
3,10681,WALL·E,8.129,21213,19.0410,7,3,4
4,1422,The Departed,8.160,16955,18.7587,6,7,5
5,1124,The Prestige,8.212,18487,18.4127,4,5,6
6,77,Memento,8.175,17065,14.8529,5,6,7
7,641,Requiem for a Dream,8.007,11500,5.3724,8,8,8


In [33]:
# Define the five provisional standalone recommendations.
selected_titles = [
    "Interstellar",
    "Inception",
    "Spirited Away",
    "WALL·E",
    "The Departed"
]

selected_ids = set(
    candidate_profiles.loc[
        candidate_profiles["title"].isin(selected_titles),
        "id"
    ]
)

# Identify recurring directors across the selected films.
selected_directors = film_directors[
    film_directors["movie_id"].isin(selected_ids)
]

director_overlap = (
    selected_directors.groupby(
        ["person_id", "person_name"]
    )
    .agg(film_count=("movie_id", "nunique"))
    .reset_index()
    .sort_values("film_count", ascending=False)
)

# Identify recurring lead cast across the selected films.
selected_cast = film_cast[
    (film_cast["movie_id"].isin(selected_ids)) &
    (film_cast["billing_order"] <= 4)
]

cast_overlap = (
    selected_cast.groupby(
        ["person_id", "person_name"]
    )
    .agg(film_count=("movie_id", "nunique"))
    .reset_index()
    .sort_values("film_count", ascending=False)
)

print("Director overlap:")
display(director_overlap)

print("Lead cast overlap:")
display(cast_overlap)

Director overlap:


,person_id,person_name,film_count
1,525,Christopher Nolan,2
0,7,Andrew Stanton,1
2,608,Hayao Miyazaki,1
3,1032,Martin Scorsese,1


Lead cast overlap:


,person_id,person_name,film_count
8,6193,Leonardo DiCaprio,2
0,514,Jack Nicholson,1
2,1813,Anne Hathaway,1
1,670,Ben Burtt,1
4,1893,Casey Affleck,1
5,2524,Tom Hardy,1
6,3895,Michael Caine,1
3,1892,Matt Damon,1
7,3899,Ken Watanabe,1
9,7907,John Ratzenberger,1


In [34]:
# Filter franchise analysis to films released from 2000 onward.
recent_franchise_films = franchise_analysis[
    (franchise_analysis["release_year"] >= 2000) &
    (franchise_analysis["collection_id_x"].notna())
].copy()

# Summarise recent films within each TMDb collection.
recent_franchise_summary = (
    recent_franchise_films
    .groupby(
        ["collection_id_x", "collection_name_x"],
        as_index=False
    )
    .agg(
        film_count=("id", "nunique"),
        average_rating=("vote_average", "mean"),
        average_popularity=("popularity", "mean"),
        average_vote_count=("vote_count", "mean"),
        median_budget=("budget", "median"),
        median_revenue=("revenue", "median")
    )
    .sort_values(
        ["film_count", "average_popularity"],
        ascending=[False, False]
    )
)

display(recent_franchise_summary.round(2))

,collection_id_x,collection_name_x,film_count,average_rating,average_popularity,average_vote_count,median_budget,median_revenue
8,1241.0,Harry Potter Collection,8,7.82,28.19,23606.75,140000000.0,9.360860e+08
4,295.0,Pirates of the Caribbean Collection,5,7.15,16.64,17153.20,230000000.0,9.616912e+08
17,86311.0,The Avengers Collection,4,7.95,51.71,32161.25,267500000.0,1.785615e+09
10,2150.0,Shrek Collection,4,6.96,14.90,12996.00,155000000.0,7.804555e+08
18,87359.0,Mission: Impossible Collection,4,6.81,8.85,9317.75,147500000.0,6.145524e+08
2,119.0,The Lord of the Rings Collection,3,8.48,34.31,27260.67,93000000.0,9.262874e+08
3,263.0,The Dark Knight Collection,3,8.01,30.45,28774.67,185000000.0,1.008481e+09
20,131292.0,Iron Man Collection,3,7.15,22.72,25549.67,200000000.0,6.239333e+08
19,121938.0,The Hobbit Collection,3,7.44,18.70,17065.00,250000000.0,9.584000e+08
13,8354.0,Ice Age Collection,3,6.97,13.22,11571.33,80000000.0,6.609988e+08


In [35]:
# Inspect the exact sampled titles in the three shortlisted collections.
shortlisted_collections = [
    "The Avengers Collection",
    "The Lord of the Rings Collection",
    "Harry Potter Collection",
]

shortlist_members = (
    recent_franchise_films[
        recent_franchise_films["collection_name_x"].isin(shortlisted_collections)
    ][
        [
            "collection_name_x",
            "title",
            "release_year",
            "vote_average",
            "vote_count",
            "popularity",
        ]
    ]
    .sort_values(["collection_name_x", "release_year"])
    .reset_index(drop=True)
)

display(shortlist_members)

,collection_name_x,title,release_year,vote_average,vote_count,popularity
0,Harry Potter Collection,Harry Potter and the Philosopher's Stone,2001,7.905,30653,41.3650
1,Harry Potter Collection,Harry Potter and the Chamber of Secrets,2002,7.710,24428,28.3974
2,Harry Potter Collection,Harry Potter and the Prisoner of Azkaban,2004,8.014,24006,27.1101
3,Harry Potter Collection,Harry Potter and the Goblet of Fire,2005,7.804,22958,31.3937
4,Harry Potter Collection,Harry Potter and the Order of the Phoenix,2007,7.673,21508,23.0685
5,Harry Potter Collection,Harry Potter and the Half-Blood Prince,2009,7.681,21468,24.1262
6,Harry Potter Collection,Harry Potter and the Deathly Hallows: Part 1,2010,7.732,21070,25.2917
7,Harry Potter Collection,Harry Potter and the Deathly Hallows: Part 2,2011,8.081,22763,24.7822
8,The Avengers Collection,The Avengers,2012,8.085,40407,41.3189
9,The Avengers Collection,Avengers: Age of Ultron,2015,7.278,25373,32.5408


In [36]:
# Build a film-level comparison using the existing franchise classification.
comparison_df = franchise_analysis[
    [
        "id",
        "release_year",
        "collection_id_x",
        "vote_average",
        "vote_count",
        "popularity",
        "budget",
        "revenue",
    ]
].copy()

# Classify each film using TMDb collection membership.
comparison_df["acquisition_type"] = comparison_df[
    "collection_id_x"
].notna().map({
    True: "Franchise",
    False: "Standalone"
})

# Define the two required comparison windows.
comparison_df["period"] = "All years"
comparison_df.loc[
    comparison_df["release_year"] >= 2000,
    "period"
] = "2000 onward"

# Calculate comparable film-level indicators.
comparison_summary = (
    comparison_df
    .groupby(["period", "acquisition_type"], as_index=False)
    .agg(
        film_count=("id", "nunique"),
        average_rating=("vote_average", "mean"),
        median_rating=("vote_average", "median"),
        average_vote_count=("vote_count", "mean"),
        median_popularity=("popularity", "median"),
        average_popularity=("popularity", "mean"),
        median_budget=("budget", "median"),
        median_revenue=("revenue", "median"),
    )
    .sort_values(["period", "acquisition_type"])
)

display(comparison_summary.round(2))

# Check that the comparison covers the expected unique films.
print("Unique films compared:", comparison_df["id"].nunique())
print("Duplicate film IDs:", comparison_df["id"].duplicated().sum())
print("\nFilm counts by period and acquisition type:")
display(
    comparison_df.groupby(
        ["period", "acquisition_type"]
    )["id"].nunique().unstack(fill_value=0)
)

,period,acquisition_type,film_count,average_rating,median_rating,average_vote_count,median_popularity,average_popularity,median_budget,median_revenue
0,2000 onward,Franchise,63,7.33,7.45,17548.79,14.88,19.35,150000000.0,786636033.0
1,2000 onward,Standalone,8,8.26,8.19,23398.00,18.90,31.01,65000000.0,283195047.5
2,All years,Franchise,33,7.71,7.86,14786.73,13.68,15.85,32350000.0,366101666.0
3,All years,Standalone,8,8.47,8.46,23083.75,26.01,27.70,25000000.0,83438541.0


Unique films compared: 112
Duplicate film IDs: 0

Film counts by period and acquisition type:


acquisition_type,Franchise,Standalone
period,,
2000 onward,63,8
All years,33,8


In [37]:
# Create independent datasets for the two required comparison windows.
all_years = comparison_df.copy()

recent_years = comparison_df[
    comparison_df["release_year"] >= 2000
].copy()

# Label each dataset explicitly.
all_years["period"] = "All years"
recent_years["period"] = "2000 onward"

# Combine the two windows without overwriting period labels.
comparison_windows = pd.concat(
    [all_years, recent_years],
    ignore_index=True
)

# Compare franchise and standalone film indicators.
comparison_summary = (
    comparison_windows
    .groupby(["period", "acquisition_type"], as_index=False)
    .agg(
        film_count=("id", "nunique"),
        average_rating=("vote_average", "mean"),
        median_rating=("vote_average", "median"),
        average_vote_count=("vote_count", "mean"),
        median_popularity=("popularity", "median"),
        average_popularity=("popularity", "mean"),
        median_budget=("budget", "median"),
        median_revenue=("revenue", "median"),
    )
    .sort_values(["period", "acquisition_type"])
)

display(comparison_summary.round(2))

# Verify the population counts for each comparison window.
display(
    comparison_windows.groupby(
        ["period", "acquisition_type"]
    )["id"].nunique().unstack(fill_value=0)
)

print("All-years records:", len(all_years))
print("Recent-years records:", len(recent_years))
print("Unique source films:", comparison_df["id"].nunique())

,period,acquisition_type,film_count,average_rating,median_rating,average_vote_count,median_popularity,average_popularity,median_budget,median_revenue
0,2000 onward,Franchise,63,7.33,7.45,17548.79,14.88,19.35,150000000.0,786636033.0
1,2000 onward,Standalone,8,8.26,8.19,23398.00,18.90,31.01,65000000.0,283195047.5
2,All years,Franchise,96,7.46,7.60,16599.33,14.67,18.15,101000000.0,578937111.0
3,All years,Standalone,16,8.37,8.44,23240.88,21.56,29.36,25000000.0,161802536.5


acquisition_type,Franchise,Standalone
period,,
2000 onward,63,8
All years,96,16


All-years records: 112
Recent-years records: 71
Unique source films: 112


In [38]:
# Count how many sampled films belong to each TMDb collection.
collection_sizes = (
    comparison_df[comparison_df["collection_id_x"].notna()]
    .groupby("collection_id_x")["id"]
    .nunique()
    .rename("sampled_collection_size")
)

# Attach collection sizes to the film-level comparison.
refined_comparison = comparison_df.merge(
    collection_sizes,
    on="collection_id_x",
    how="left",
    validate="many_to_one"
)

# Distinguish multi-film collections, single sampled collection films,
# and films without a TMDb collection.
refined_comparison["acquisition_category"] = "Standalone / no TMDb collection"

refined_comparison.loc[
    refined_comparison["sampled_collection_size"] == 1,
    "acquisition_category"
] = "Single sampled film in collection"

refined_comparison.loc[
    refined_comparison["sampled_collection_size"] > 1,
    "acquisition_category"
] = "Multi-film sampled collection"

# Create the two required time windows.
all_years_refined = refined_comparison.copy()
all_years_refined["period"] = "All years"

recent_refined = refined_comparison[
    refined_comparison["release_year"] >= 2000
].copy()
recent_refined["period"] = "2000 onward"

refined_windows = pd.concat(
    [all_years_refined, recent_refined],
    ignore_index=True
)

# Compare the three categories.
refined_summary = (
    refined_windows
    .groupby(["period", "acquisition_category"], as_index=False)
    .agg(
        film_count=("id", "nunique"),
        average_rating=("vote_average", "mean"),
        average_popularity=("popularity", "mean"),
        average_vote_count=("vote_count", "mean"),
        median_revenue=("revenue", "median")
    )
    .sort_values(["period", "acquisition_category"])
)

display(refined_summary.round(2))

# Confirm category membership and collection counts.
display(
    refined_comparison.groupby("acquisition_category").agg(
        film_count=("id", "nunique"),
        collection_count=("collection_id_x", "nunique")
    )
)

,period,acquisition_category,film_count,average_rating,average_popularity,average_vote_count,median_revenue
0,2000 onward,Multi-film sampled collection,59,7.33,19.62,17565.53,795922298.0
1,2000 onward,Single sampled film in collection,4,7.25,15.38,17302.00,597595707.5
2,2000 onward,Standalone / no TMDb collection,8,8.26,31.01,23398.00,283195047.5
3,All years,Multi-film sampled collection,87,7.44,18.57,16634.01,618638999.0
4,All years,Single sampled film in collection,9,7.63,14.12,16264.11,470653000.0
5,All years,Standalone / no TMDb collection,16,8.37,29.36,23240.88,161802536.5


,film_count,collection_count
acquisition_category,,
Multi-film sampled collection,87,23
Single sampled film in collection,9,9
Standalone / no TMDb collection,16,0


In [40]:
from pathlib import Path

audit_files = list((PROJECT_ROOT / "out").glob("*audit*.csv"))

for path in audit_files:
    print(path)

In [41]:
# Load the original candidate list.
id_list = pd.read_csv(
    PROJECT_ROOT / "data" / "raw" / "id_list.csv",
    dtype={"tmdb_id": "string"},
    keep_default_na=False
)

# Classify candidate IDs.
id_list["is_numeric"] = id_list["tmdb_id"].str.fullmatch(r"[+-]?\d+")
id_list["is_positive"] = (
    id_list["is_numeric"]
    & pd.to_numeric(id_list["tmdb_id"], errors="coerce").gt(0)
)

valid_ids = id_list.loc[id_list["is_positive"], "tmdb_id"]

# Reconcile source rows against retrieved film records.
print("Original source rows:", len(id_list))
print("Unique valid positive IDs:", valid_ids.nunique())
print("Invalid source rows:", (~id_list["is_positive"]).sum())
print("Duplicate excess occurrences:", valid_ids.duplicated().sum())
print("Successfully retrieved films:", films_analysis["id"].nunique())
print("Valid IDs not retrieved:", valid_ids.nunique() - films_analysis["id"].nunique())

Original source rows: 126
Unique valid positive IDs: 117
Invalid source rows: 3
Duplicate excess occurrences: 6
Successfully retrieved films: 112
Valid IDs not retrieved: 5


In [42]:
display(id_list.loc[~id_list["is_positive"], ["tmdb_id"]])

,tmdb_id
28,-1
92,tt0468569
108,N/A


In [43]:
# Identify existing talent-related DataFrames.
for name, value in list(globals().items()):
    if isinstance(value, pd.DataFrame) and any(
        term in name.lower()
        for term in ["lead", "actor", "director", "talent"]
    ):
        print(f"{name}: {value.shape}")

film_directors: (122, 6)
cast_talent_summary: (330, 7)
director_talent_summary: (66, 6)
writer_talent_summary: (160, 6)
producer_talent_summary: (277, 6)
selected_directors: (5, 6)
director_overlap: (4, 3)


In [44]:
# Inspect the director summary structure and its strongest repeated appearances.
display(
    director_talent_summary
    .sort_values(
        ["film_count", "average_rating"],
        ascending=[False, False]
    )
    .head(15)
)

# Inspect the underlying film-level director records.
display(film_directors.head(10))

KeyError: 'average_rating'

In [45]:
print("Director summary columns:")
print(director_talent_summary.columns.tolist())

print("\nDirector summary sample:")
display(director_talent_summary.head(10))

print("\nFilm director columns:")
print(film_directors.columns.tolist())
display(film_directors.head(5))

Director summary columns:
['person_id', 'person_name', 'film_count', 'average_popularity', 'median_popularity', 'maximum_popularity']

Director summary sample:


,person_id,person_name,film_count,average_popularity,median_popularity,maximum_popularity
6,488,Steven Spielberg,8,4.96,4.96,4.96
7,525,Christopher Nolan,7,8.27,8.27,8.27
4,108,Peter Jackson,6,2.49,2.49,2.49
0,1,George Lucas,4,2.37,2.37,2.37
3,24,Robert Zemeckis,4,2.08,2.08,2.08
40,11343,David Yates,4,1.22,1.22,1.22
5,138,Quentin Tarantino,3,4.94,4.94,4.94
19,2710,James Cameron,3,3.46,3.46,3.46
16,1776,Francis Ford Coppola,3,1.89,1.89,1.89
17,1884,Steven Soderbergh,3,1.64,1.64,1.64



Film director columns:
['movie_id', 'person_id', 'person_name', 'department', 'job', 'person_popularity']


,movie_id,person_id,person_name,department,job,person_popularity
0,10138,15277,Jon Favreau,Directing,Director,2.9450
1,10192,64151,Mike Mitchell,Directing,Director,0.7373
2,10193,8,Lee Unkrich,Directing,Director,0.6138
3,105,24,Robert Zemeckis,Directing,Director,2.0826
4,10681,7,Andrew Stanton,Directing,Director,1.6447


In [46]:
# Join each director to the film's audience indicators.
director_performance = film_directors.merge(
    films_analysis[
        ["id", "vote_average", "vote_count", "popularity"]
    ],
    left_on="movie_id",
    right_on="id",
    how="left",
    validate="many_to_one"
)

# Summarise the performance of films associated with each director.
director_performance_summary = (
    director_performance
    .groupby(["person_id", "person_name"], as_index=False)
    .agg(
        film_count=("movie_id", "nunique"),
        average_rating=("vote_average", "mean"),
        minimum_rating=("vote_average", "min"),
        average_vote_count=("vote_count", "mean"),
        average_film_popularity=("popularity", "mean")
    )
    .sort_values(
        ["film_count", "average_rating"],
        ascending=[False, False]
    )
)

display(director_performance_summary.head(15))

# Confirm that every director record matched a film.
print("Director records:", len(film_directors))
print("Unmatched film records:", director_performance["vote_average"].isna().sum())

,person_id,person_name,film_count,average_rating,minimum_rating,average_vote_count,average_film_popularity
6,488,Steven Spielberg,8,7.490250,6.027,13028.875000,11.491912
7,525,Christopher Nolan,7,8.184857,7.733,29227.857143,35.231943
4,108,Peter Jackson,6,7.958667,7.342,22162.833333,26.505767
3,24,Robert Zemeckis,4,8.018000,7.494,20067.750000,17.933125
40,11343,David Yates,4,7.791750,7.673,21702.250000,24.317150
0,1,George Lucas,4,7.216000,6.582,17518.250000,14.764700
16,1776,Francis Ford Coppola,3,8.225000,7.413,15285.000000,27.288167
5,138,Quentin Tarantino,3,8.107000,7.871,22270.333333,18.681067
19,2710,James Cameron,3,7.949333,7.700,14020.000000,18.311800
15,1704,Gore Verbinski,3,7.503667,7.280,19023.333333,18.555267


Director records: 122
Unmatched film records: 0
